# 🧠 Exercícios — Redes Neurais Artificiais

**Disciplina:** Inteligência Artificial | **Nível:** Avançado

> Implemente e treine redes neurais do zero usando backpropagation. Resolva problemas clássicos como XOR e espirais.


## 1. Perceptron e a Função AND

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
def sigmoid_deriv(z): return sigmoid(z)*(1-sigmoid(z))
def relu(z): return np.maximum(0, z)
def relu_deriv(z): return (z > 0).astype(float)

# AND
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([[0],[0],[0],[1]], dtype=float)

class PerceptronSingle:
    def __init__(self, lr=0.1):
        np.random.seed(42)
        self.W = np.random.randn(2,1)*0.1
        self.b = 0.0; self.lr = lr
    
    def forward(self, X): return sigmoid(X@self.W+self.b)
    
    def treinar(self, X, y, epocas=2000):
        historico = []
        for _ in range(epocas):
            yp = self.forward(X)
            loss = np.mean((yp-y)**2)
            dL = (yp-y)*sigmoid_deriv(X@self.W+self.b)
            self.W -= self.lr*(X.T@dL)/len(X)
            self.b -= self.lr*dL.mean()
            historico.append(loss)
        return historico

p = PerceptronSingle()
hist = p.treinar(X, y_and)
plt.figure(figsize=(8,3)); plt.plot(hist); plt.title('Loss — AND'); plt.xlabel('Época'); plt.ylabel('MSE'); plt.grid(True); plt.show()
print("AND com 1 neurônio:")
for xi, yi in zip(X, y_and):
    pred = p.forward(xi.reshape(1,-1))[0,0]
    print(f"  {xi} → {pred:.3f} ({'✅' if round(pred)==yi[0] else '❌'})")


## 2. Rede Neural com 1 Camada Oculta — Problema XOR

In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

class RedeNeural:
    """Rede com 1 camada oculta: entrada → oculta → saída."""
    
    def __init__(self, n_entrada, n_oculta, n_saida, lr=0.5):
        np.random.seed(42)
        self.W1 = np.random.randn(n_entrada, n_oculta)*0.5
        self.b1 = np.zeros((1, n_oculta))
        self.W2 = np.random.randn(n_oculta, n_saida)*0.5
        self.b2 = np.zeros((1, n_saida))
        self.lr = lr
    
    def forward(self, X):
        self.z1 = X@self.W1+self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = self.a1@self.W2+self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2
    
    def backward(self, X, y):
        m = len(X)
        dL_da2 = (self.a2-y)
        da2_dz2 = sigmoid_deriv(self.z2)
        delta2 = dL_da2*da2_dz2
        
        dW2 = self.a1.T@delta2/m
        db2 = delta2.mean(axis=0, keepdims=True)
        
        delta1 = (delta2@self.W2.T)*sigmoid_deriv(self.z1)
        dW1 = X.T@delta1/m
        db1 = delta1.mean(axis=0, keepdims=True)
        
        self.W2 -= self.lr*dW2; self.b2 -= self.lr*db2
        self.W1 -= self.lr*dW1; self.b1 -= self.lr*db1
    
    def treinar(self, X, y, epocas=5000):
        historico = []
        for _ in range(epocas):
            self.forward(X)
            self.backward(X, y)
            historico.append(np.mean((self.a2-y)**2))
        return historico

rn = RedeNeural(2, 4, 1, lr=0.8)
hist_xor = rn.treinar(X_xor, y_xor, epocas=5000)

plt.figure(figsize=(8,3)); plt.plot(hist_xor); plt.title('Loss — XOR'); plt.xlabel('Época'); plt.ylabel('MSE'); plt.grid(True); plt.show()
print("XOR com rede neural (2 camadas):")
for xi, yi in zip(X_xor, y_xor):
    pred = rn.forward(xi.reshape(1,-1))[0,0]
    print(f"  {xi} → {pred:.3f} ({'✅' if round(pred)==yi[0] else '❌'})")


### 📝 Exercício 1

Experimente com **diferentes números de neurônios ocultos** (2, 4, 8, 16). Quantos são necessários para resolver o XOR? Há diferença na velocidade de convergência?

In [ ]:
for n_oculta in [2, 4, 8, 16]:
    rn_teste = RedeNeural(2, n_oculta, 1, lr=0.8)
    hist = rn_teste.treinar(X_xor, y_xor, epocas=3000)
    preds = rn_teste.forward(X_xor).flatten()
    acertos = sum(round(p)==int(y[0]) for p,y in zip(preds, y_xor))
    print(f"Oculta={n_oculta}: acertos={acertos}/4, loss_final={hist[-1]:.5f}")


## 3. Classificação de Espirais — Problema Não-Linear

In [ ]:
# Geração do dataset de espirais
def gerar_espirais(n=100, voltas=1.5):
    np.random.seed(42)
    theta = np.sqrt(np.random.rand(n))*voltas*2*np.pi
    r = theta + np.random.randn(n)*0.4
    X1 = np.c_[r*np.cos(theta), r*np.sin(theta)]
    r2 = theta + np.random.randn(n)*0.4
    X2 = np.c_[-r2*np.cos(theta), -r2*np.sin(theta)]
    X = np.vstack([X1, X2])
    y = np.array([[0]]*n + [[1]]*n, dtype=float)
    # Normalizar
    X = (X - X.mean(axis=0)) / X.std(axis=0)
    return X, y

X_esp, y_esp = gerar_espirais()

plt.figure(figsize=(6,6))
plt.scatter(X_esp[:100,0],X_esp[:100,1],c='blue',s=20,label='Classe 0')
plt.scatter(X_esp[100:,0],X_esp[100:,1],c='red',s=20,label='Classe 1')
plt.title('Dataset de Espirais'); plt.legend(); plt.grid(True); plt.show()

# Treinar rede neural de 2 camadas ocultas
class RedeNeural2Ocultas:
    def __init__(self, n0, n1, n2, n3, lr=0.01):
        np.random.seed(42)
        self.W1 = np.random.randn(n0,n1)*np.sqrt(2/n0)
        self.b1 = np.zeros((1,n1))
        self.W2 = np.random.randn(n1,n2)*np.sqrt(2/n1)
        self.b2 = np.zeros((1,n2))
        self.W3 = np.random.randn(n2,n3)*np.sqrt(2/n2)
        self.b3 = np.zeros((1,n3))
        self.lr = lr
    
    def forward(self, X):
        self.z1=X@self.W1+self.b1; self.a1=relu(self.z1)
        self.z2=self.a1@self.W2+self.b2; self.a2=relu(self.z2)
        self.z3=self.a2@self.W3+self.b3; self.a3=sigmoid(self.z3)
        return self.a3
    
    def backward(self, X, y):
        m=len(X)
        d3=(self.a3-y)*sigmoid_deriv(self.z3)
        dW3=self.a2.T@d3/m; db3=d3.mean(0,keepdims=True)
        d2=(d3@self.W3.T)*relu_deriv(self.z2)
        dW2=self.a1.T@d2/m; db2=d2.mean(0,keepdims=True)
        d1=(d2@self.W2.T)*relu_deriv(self.z1)
        dW1=X.T@d1/m; db1=d1.mean(0,keepdims=True)
        for W,b,dW,db in [(self.W3,self.b3,dW3,db3),(self.W2,self.b2,dW2,db2),(self.W1,self.b1,dW1,db1)]:
            W-=self.lr*dW; b-=self.lr*db
    
    def treinar(self, X, y, epocas=2000):
        h=[]
        for _ in range(epocas):
            self.forward(X); self.backward(X,y)
            h.append(np.mean((self.a3-y)**2))
        return h

rn2 = RedeNeural2Ocultas(2,32,16,1,lr=0.05)
h2 = rn2.treinar(X_esp, y_esp, epocas=3000)
preds = (rn2.forward(X_esp)>0.5).astype(int).flatten()
acc = (preds == y_esp.flatten().astype(int)).mean()
print(f"Acurácia nas espirais: {acc:.2%}")
plt.figure(figsize=(8,3)); plt.plot(h2); plt.title('Loss — Espirais'); plt.xlabel('Época'); plt.grid(True); plt.show()


### 📝 Exercício Final

Visualize as **fronteiras de decisão** da rede treinada nas espirais. Perceba como a rede aprendeu uma fronteira não-linear complexa.

In [ ]:
h = 0.05
x_min,x_max = X_esp[:,0].min()-0.5, X_esp[:,0].max()+0.5
y_min2,y_max = X_esp[:,1].min()-0.5, X_esp[:,1].max()+0.5
xx,yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min2,y_max,h))
Z = (rn2.forward(np.c_[xx.ravel(),yy.ravel()])>0.5).reshape(xx.shape)
plt.figure(figsize=(7,7))
plt.contourf(xx,yy,Z,alpha=0.3,cmap='bwr')
plt.scatter(X_esp[:100,0],X_esp[:100,1],c='blue',s=25,label='Classe 0')
plt.scatter(X_esp[100:,0],X_esp[100:,1],c='red',s=25,label='Classe 1')
plt.title(f'Fronteira de Decisão — RNA (acurácia={acc:.0%})')
plt.legend(); plt.grid(True); plt.show()
